# 20 — Resolve CDISC CT for COSMoS Graph

Step 2b of the cosmos-bc-dss rewrite — resolves the concept IDs carried by the structural flatten (`Codelists.codelist_concept_id`, `Variables.assigned_term_concept_id`) against the NCI EVS SDTM Controlled Terminology package.

Adds semantic content (submission values, synonyms, definitions, NCI preferred terms) as separate lookup sheets. The `Variables` and `Codelists` columns keep their authored C-codes unchanged — consumers join on ID.

**Input**
- `interim/COSMoS_Graph.xlsx` — output of notebook 10
- `downloads/SDTM_Terminology.txt` — NCI EVS SDTM CT package (tab-separated, 8 columns, ~47k rows)

**Output**
- `interim/COSMoS_Graph_CT.xlsx` with sheets: `ReadMe`, `Codelists`, `CodelistTerms`, `AssignedTerms`, `Unresolved`

**Out of scope** — NCIt enrichment beyond what the SDTM CT package carries (UMLS mappings, hierarchy, cross-ontology identity) is Step 3. Inline `value_list` resolution and `subset_codelist` parsing stay as-authored (Step 2 design decisions).

Design rationale: `cosmos-bc-dss/docs/COSMoS_Flattener_Rewrite.md`.


## 1. Imports


In [ ]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd

print(f'python : {sys.version.split()[0]}')
print(f'pandas : {pd.__version__}')


## 2. Configuration


In [ ]:
REPO_ROOT = Path.cwd().parent.parent  # notebooks/ -> cosmos-bc-dss/ -> repo root

DOWNLOADS = REPO_ROOT / 'cosmos-bc-dss' / 'downloads'
INTERIM   = REPO_ROOT / 'cosmos-bc-dss' / 'interim'

GRAPH_XLSX = INTERIM / 'COSMoS_Graph.xlsx'
CT_TXT     = DOWNLOADS / 'SDTM_Terminology.txt'
OUT_XLSX   = INTERIM / 'COSMoS_Graph_CT.xlsx'

for p in (GRAPH_XLSX, CT_TXT):
    assert p.exists(), f'missing: {p}'

print(f'in  : {GRAPH_XLSX.relative_to(REPO_ROOT)}')
print(f'CT  : {CT_TXT.relative_to(REPO_ROOT)}')
print(f'out : {OUT_XLSX.relative_to(REPO_ROOT)}')


## 3. Load the structural graph

The two sheets we need to enrich: `Codelists` (referenced codelist bindings) and `Variables` (source of pinned `assigned_term_concept_id` values).


In [ ]:
codelists_in  = pd.read_excel(GRAPH_XLSX, sheet_name='Codelists')
variables_in  = pd.read_excel(GRAPH_XLSX, sheet_name='Variables')

print(f'Codelists : {codelists_in.shape}')
print(f'Variables : {variables_in.shape}')

referenced_codelist_ids = set(codelists_in['codelist_concept_id'].dropna().unique())
referenced_term_ids     = set(variables_in['assigned_term_concept_id'].dropna().unique())
print(f'\ndistinct codelist C-codes referenced : {len(referenced_codelist_ids):,}')
print(f'distinct assigned-term C-codes         : {len(referenced_term_ids):,}')


## 4. Load SDTM CT package

The NCI EVS file has a single table with two row types distinguished by whether `Codelist Code` is populated:

- **Codelist-level rows** — `Code` is the codelist C-code, `Codelist Code` is empty. `Codelist Name`, `Codelist Extensible`, and `NCI Preferred Term` describe the codelist itself.
- **Term-level rows** — `Code` is a term C-code, `Codelist Code` is the parent codelist's C-code. `CDISC Submission Value` is the term's submission value; `CDISC Synonym(s)` and `CDISC Definition` describe the term.

We split the table on this structural distinction and then join against our references.


In [ ]:
ct_raw = pd.read_csv(CT_TXT, sep='\t', dtype=str, keep_default_na=False, na_values=[''])
print(f'SDTM_Terminology.txt : {ct_raw.shape}')
print(f'columns              : {list(ct_raw.columns)}')

# Rename to canonical snake_case
CT_RENAME = {
    'Code':                         'concept_id',
    'Codelist Code':                'parent_codelist_concept_id',
    'Codelist Extensible (Yes/No)': 'codelist_extensible',
    'Codelist Name':                'codelist_name',
    'CDISC Submission Value':       'submission_value',
    'CDISC Synonym(s)':             'synonyms',
    'CDISC Definition':             'definition',
    'NCI Preferred Term':           'nci_preferred_term',
}
ct = ct_raw.rename(columns=CT_RENAME)

is_codelist_row = ct['parent_codelist_concept_id'].isna()
ct_codelist = ct.loc[ is_codelist_row].copy()
ct_term     = ct.loc[~is_codelist_row].copy()

print(f'\ncodelist-level rows : {len(ct_codelist):,}')
print(f'term-level rows     : {len(ct_term):,}')


## 5. Build `Codelists` sheet — codelist-level enrichment

Join each referenced codelist to its CT codelist-level row to pick up `codelist_name`, `codelist_extensible`, and `codelist_nci_preferred_term`. The input `codelist_submission_value` is preserved as authored; the CT-derived submission value is added alongside for cross-check.


In [ ]:
codelist_enrich = ct_codelist[[
    'concept_id', 'submission_value', 'codelist_name',
    'codelist_extensible', 'nci_preferred_term',
]].rename(columns={
    'concept_id':         'codelist_concept_id',
    'submission_value':   'codelist_submission_value_ct',
    'nci_preferred_term': 'codelist_nci_preferred_term',
})

codelists_sheet = codelists_in.merge(codelist_enrich, on='codelist_concept_id', how='left')

# Consistency check — where CT resolved, authored submission_value should match CT submission_value
resolved = codelists_sheet['codelist_submission_value_ct'].notna()
mismatch = resolved & (
    codelists_sheet['codelist_submission_value']
    != codelists_sheet['codelist_submission_value_ct']
)

print(f'Codelists sheet: {codelists_sheet.shape}')
print(f'resolved: {resolved.sum():,}  unresolved: {(~resolved).sum():,}')
print(f'submission_value mismatches (authored vs CT): {mismatch.sum()}')
if mismatch.any():
    print(codelists_sheet.loc[mismatch, [
        'codelist_concept_id', 'codelist_submission_value', 'codelist_submission_value_ct',
    ]].to_string())
codelists_sheet.head()


## 6. Build `CodelistTerms` sheet — full permissible value sets

For every referenced codelist, expand to all its terms. Columns: codelist identity (`codelist_concept_id`, `codelist_submission_value`) joined with term identity (`term_concept_id`, `term_submission_value`, `term_synonyms`, `term_definition`, `term_nci_preferred_term`).

Grain: one row per (codelist, term). A consumer answering 'what values can this DSS bind?' joins `Variables.codelist_concept_id` → `CodelistTerms.codelist_concept_id`.


In [ ]:
terms_for_referenced = ct_term[
    ct_term['parent_codelist_concept_id'].isin(referenced_codelist_ids)
].copy()

# Bring in codelist-level submission value for convenience
cl_sub = ct_codelist.set_index('concept_id')['submission_value'].to_dict()
terms_for_referenced['codelist_submission_value'] = (
    terms_for_referenced['parent_codelist_concept_id'].map(cl_sub)
)

codelist_terms_sheet = terms_for_referenced[[
    'parent_codelist_concept_id', 'codelist_submission_value',
    'concept_id', 'submission_value', 'synonyms', 'definition', 'nci_preferred_term',
]].rename(columns={
    'parent_codelist_concept_id': 'codelist_concept_id',
    'concept_id':                 'term_concept_id',
    'submission_value':           'term_submission_value',
    'synonyms':                   'term_synonyms',
    'definition':                 'term_definition',
    'nci_preferred_term':         'term_nci_preferred_term',
}).sort_values(['codelist_concept_id', 'term_submission_value']).reset_index(drop=True)

print(f'CodelistTerms sheet: {codelist_terms_sheet.shape}')
print(f'distinct codelists covered: {codelist_terms_sheet.codelist_concept_id.nunique()}')
avg_terms = codelist_terms_sheet.groupby('codelist_concept_id').size().mean()
print(f'average terms per codelist: {avg_terms:.1f}')
codelist_terms_sheet.head()


## 7. Build `AssignedTerms` sheet — pinned term identity (context-stable fields only)

For each distinct `assigned_term_concept_id` referenced by `Variables`, resolve the term-level identity.

**Grain**: one row per distinct `assigned_term_concept_id`.

**Columns**: only fields that are stable per C-code across codelist contexts — `term_definition`, `term_nci_preferred_term`. The context-variant `submission_value` and `synonyms` are deliberately NOT included here: the same C-code can carry different CDISC submission values in different codelists (e.g., `Length` vs `LENGTH` for TEST vs TESTCD), and synonyms often include codelist-context prefixes.

**To get context-specific submission_value / synonyms**, join `Variables` to `CodelistTerms` on both `codelist_concept_id` AND `assigned_term_concept_id` (= `term_concept_id`).


In [ ]:
# Count variable uses per pinned term
term_usage = (
    variables_in['assigned_term_concept_id']
    .dropna()
    .value_counts()
    .rename_axis('assigned_term_concept_id')
    .reset_index(name='used_in_variables_count')
)

# Stable-per-C-code fields only. definition and nci_preferred_term are invariant across
# codelist contexts (verified empirically — 0 within-code variance on the 2026-Q1 data).
# submission_value and synonyms vary by codelist context and belong in CodelistTerms.
assigned_enrich = (
    ct[['concept_id', 'definition', 'nci_preferred_term']]
    .rename(columns={
        'concept_id':         'assigned_term_concept_id',
        'definition':         'term_definition',
        'nci_preferred_term': 'term_nci_preferred_term',
    })
    .drop_duplicates(subset=['assigned_term_concept_id'])
)

assigned_terms_sheet = term_usage.merge(
    assigned_enrich, on='assigned_term_concept_id', how='left'
).sort_values('used_in_variables_count', ascending=False).reset_index(drop=True)

assert assigned_terms_sheet['assigned_term_concept_id'].is_unique, 'non-unique pinned term C-codes after dedup'

resolved = assigned_terms_sheet['term_definition'].notna()
print(f'AssignedTerms sheet: {assigned_terms_sheet.shape}')
print(f'resolved: {resolved.sum():,}  unresolved: {(~resolved).sum():,}')
assigned_terms_sheet.head(10)


## 8. Build `Unresolved` sheet — gaps for human review

Any `codelist_concept_id` or `assigned_term_concept_id` that the SDTM CT package does not know about. Does not block downstream use — just flagged.


In [ ]:
ct_known_ids = set(ct['concept_id'].dropna().unique())

unresolved_codelists = [
    {'source': 'Codelists', 'concept_id': c, 'context': 'codelist_concept_id'}
    for c in sorted(referenced_codelist_ids - ct_known_ids)
]
unresolved_terms = [
    {'source': 'Variables', 'concept_id': c, 'context': 'assigned_term_concept_id'}
    for c in sorted(referenced_term_ids - ct_known_ids)
]
unresolved_sheet = pd.DataFrame(unresolved_codelists + unresolved_terms)

# Annotate usage so a reviewer can see impact
use_map_cl = dict(zip(codelists_in['codelist_concept_id'], codelists_in['variable_uses_count']))
use_map_tm = dict(zip(term_usage['assigned_term_concept_id'], term_usage['used_in_variables_count']))
def _uses(row):
    if row['source'] == 'Codelists':
        return use_map_cl.get(row['concept_id'])
    return use_map_tm.get(row['concept_id'])
if len(unresolved_sheet):
    unresolved_sheet['variable_uses_count'] = unresolved_sheet.apply(_uses, axis=1)

print(f'Unresolved sheet: {unresolved_sheet.shape}')
if len(unresolved_sheet):
    print(unresolved_sheet.to_string())


## 9. Build `Anomalies` sheet — pinned-term-not-in-bound-codelist

Some variables are pinned to a term that is NOT a permissible value of the codelist the variable is bound to. This is a data anomaly in the source — flagged here, not fixed.

Scope: only variables where both `codelist_concept_id` and `assigned_term_concept_id` are populated AND both resolve against the CT package. Rows where one or both IDs are unresolved are already in the `Unresolved` sheet.


In [ ]:
# Set of valid (codelist, term) memberships from CT
valid_pairs = set(zip(
    codelist_terms_sheet['codelist_concept_id'],
    codelist_terms_sheet['term_concept_id'],
))

# Variables rows with both IDs populated and both known to CT
vpair = variables_in[
    variables_in['codelist_concept_id'].notna()
    & variables_in['assigned_term_concept_id'].notna()
    & variables_in['codelist_concept_id'].isin(ct_known_ids)
    & variables_in['assigned_term_concept_id'].isin(ct_known_ids)
].copy()

vpair['is_member'] = [
    (c, t) in valid_pairs
    for c, t in zip(vpair['codelist_concept_id'], vpair['assigned_term_concept_id'])
]

anomalies_sheet = vpair.loc[~vpair['is_member'], [
    'ds_id', 'variable_name',
    'codelist_concept_id', 'codelist_submission_value',
    'assigned_term_concept_id', 'assigned_term_value',
]].copy()
anomalies_sheet.insert(0, 'issue_type', 'pinned_term_not_in_bound_codelist')
anomalies_sheet = anomalies_sheet.reset_index(drop=True)

print(f'Anomalies sheet: {anomalies_sheet.shape}')
if len(anomalies_sheet):
    print(anomalies_sheet.to_string())


## 10. Resolution summary

Coverage stats surfaced in the ReadMe.


In [ ]:
summary = {
    'codelists_referenced':        len(referenced_codelist_ids),
    'codelists_resolved':           int(codelists_sheet['codelist_submission_value_ct'].notna().sum()),
    'codelists_unresolved':         len(referenced_codelist_ids - ct_known_ids),
    'assigned_terms_distinct':      len(referenced_term_ids),
    'assigned_terms_resolved':      int(assigned_terms_sheet['term_definition'].notna().sum()),
    'assigned_terms_unresolved':    len(referenced_term_ids - ct_known_ids),
    'codelist_terms_total_rows':    len(codelist_terms_sheet),
    'submission_value_mismatches':  int(mismatch.sum()),
    'pinned_term_not_in_codelist':  len(anomalies_sheet),
}
for k, v in summary.items():
    print(f'  {k:32s} : {v:>6,}')


## 11. Write `interim/COSMoS_Graph_CT.xlsx`


In [ ]:
readme_lines = [
    'COSMoS Graph — CT-resolved view',
    '',
    f'Generated : {date.today().isoformat()}',
    'Pipeline  : cosmos-bc-dss/notebooks/20_resolve_ct.ipynb',
    f'Input     : {GRAPH_XLSX.name}, {CT_TXT.name}',
    '',
    'SHEETS',
    f'  Codelists       {len(codelists_sheet):>6,} rows — input bindings + CT enrichment',
    f'  CodelistTerms   {len(codelist_terms_sheet):>6,} rows — one per (codelist, term) for referenced codelists',
    f'  AssignedTerms   {len(assigned_terms_sheet):>6,} rows — resolved identity for pinned terms (stable fields only)',
    f'  Unresolved      {len(unresolved_sheet):>6,} rows — concept IDs not in SDTM_Terminology.txt',
    f'  Anomalies       {len(anomalies_sheet):>6,} rows — pinned term not member of bound codelist',
    '',
    'COVERAGE',
]
for k, v in summary.items():
    readme_lines.append(f'  {k:32s} : {v:,}')
readme_lines += [
    '',
    'JOIN GUIDE',
    '  Variables.codelist_concept_id       -> Codelists.codelist_concept_id         (codelist identity)',
    '  Variables.codelist_concept_id       -> CodelistTerms.codelist_concept_id     (full permissible value list)',
    '  Variables.assigned_term_concept_id  -> AssignedTerms.assigned_term_concept_id (stable term identity)',
    '  Variables.(codelist_concept_id, assigned_term_concept_id)',
    '    -> CodelistTerms.(codelist_concept_id, term_concept_id)   (context-specific submission_value, synonyms)',
    '',
    'See cosmos-bc-dss/docs/COSMoS_Flattener_Rewrite.md for the pipeline design.',
    'CT source: NCI EVS SDTM Terminology package 2026-03-27.',
]

readme_df = pd.DataFrame({'README': readme_lines})

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name='ReadMe', index=False, header=False)
    codelists_sheet.to_excel(writer, sheet_name='Codelists', index=False)
    codelist_terms_sheet.to_excel(writer, sheet_name='CodelistTerms', index=False)
    assigned_terms_sheet.to_excel(writer, sheet_name='AssignedTerms', index=False)
    unresolved_sheet.to_excel(writer, sheet_name='Unresolved', index=False)
    anomalies_sheet.to_excel(writer, sheet_name='Anomalies', index=False)

print(f'wrote {OUT_XLSX}')
print(f'size : {OUT_XLSX.stat().st_size:,} bytes')


## 12. Summary


In [ ]:
xl = pd.ExcelFile(OUT_XLSX)
print(f'sheets in output: {xl.sheet_names}')
for s in xl.sheet_names:
    sh = pd.read_excel(OUT_XLSX, sheet_name=s, header=None if s == 'ReadMe' else 0)
    print(f'  {s:15s}: {sh.shape}')
